In [0]:
%sql
-- ============================================================
-- Retail Sales Metric View
-- Source  : retail.gold.fact_sales  (Gold MV)
-- Joins   : dim_calendar (Gold), product_catalog (Silver), account (Silver)
-- Purpose : Central semantic layer for revenue, pipeline, product &
--           customer KPIs — ready for AI/BI dashboards and Genie spaces
-- ============================================================
drop view retail.gold.retail_metrics_view;
drop view retail.gold.retail_sales_metrics;

CREATE OR REPLACE VIEW retail.metric_view.retail_metrics_view
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
source: retail.gold.fact_sales
comment: >
  Retail sales metric view covering revenue performance, pipeline health,
  product mix, customer segmentation, and time-series analysis.
  Designed for AI/BI dashboards and Genie spaces.
joins:
  - name: cal
    source: retail.gold.dim_calendar
    on: source.transaction_date = cal.calendar_date
    cardinality: many_to_one
    rely:
      at_most_one_match: true
  - name: prod
    source: retail.silver.product_catalog
    on: source.product_id = prod.product_id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
  - name: acct
    source: retail.silver.account
    on: source.account_id = acct.id
    cardinality: many_to_one
    rely:
      at_most_one_match: true
dimensions:
  # ── Time dimensions (from dim_calendar) ─────────────────────
  - name: Transaction Date
    expr: source.transaction_date
    display_name: Transaction Date
    comment: Date the sales transaction was recorded
    synonyms:
      - sale date
      - order date
    format:
      type: date
      date_format: locale_short_month
  - name: Year
    expr: cal.year
    display_name: Year
    comment: Calendar year of the transaction
    synonyms:
      - sales year
      - calendar year
  - name: Quarter
    expr: cal.year_quarter
    display_name: Quarter
    comment: Year and quarter label (e.g. 2024-Q1)
    synonyms:
      - year quarter
      - quarterly period
  - name: Month
    expr: cal.year_month
    display_name: Month
    comment: Year-month label (e.g. 2024-01) for time-series charts
    synonyms:
      - year month
      - sales month
  - name: Month Name
    expr: cal.month_name
    display_name: Month Name
    comment: Full calendar month name (January, February ...)
    synonyms:
      - month
  - name: Week
    expr: cal.year_week
    display_name: Week
    comment: Year and ISO week number of the transaction
    synonyms:
      - year week
  - name: Day of Week
    expr: cal.day_name
    display_name: Day of Week
    comment: Full weekday name (Monday through Sunday)
    synonyms:
      - weekday
      - day name
  - name: Is Weekend
    expr: cal.is_weekend
    display_name: Is Weekend
    comment: True if the transaction occurred on a Saturday or Sunday
    synonyms:
      - weekend transaction
  - name: Fiscal Year
    expr: cal.fiscal_year
    display_name: Fiscal Year
    comment: Fiscal year of the transaction
    synonyms:
      - FY
      - financial year
  - name: Fiscal Quarter
    expr: cal.fiscal_quarter
    display_name: Fiscal Quarter
    comment: Fiscal quarter number (1-4)
    synonyms:
      - FQ
      - fiscal period
  # ── Transaction dimensions (from fact_sales) ─────────────────
  - name: Payment Mode
    expr: source.payment_mode
    display_name: Payment Mode
    comment: Method of payment (e.g. Credit Card, Cash, Online)
    synonyms:
      - payment method
      - payment type
  - name: Sales Channel
    expr: source.sales_channel
    display_name: Sales Channel
    comment: Channel through which the sale was made (e.g. Online, In-Store)
    synonyms:
      - channel
      - distribution channel
  - name: Stage Name
    expr: source.stage_name
    display_name: Opportunity Stage
    comment: Current stage of the sales opportunity pipeline
    synonyms:
      - pipeline stage
      - opportunity stage
      - deal stage
  - name: Lead Source
    expr: source.lead_source
    display_name: Lead Source
    comment: Origin channel that generated the lead
    synonyms:
      - acquisition source
      - source
  - name: Forecast Category
    expr: source.forecast_category
    display_name: Forecast Category
    comment: Sales forecast classification (e.g. Commit, Best Case, Pipeline)
    synonyms:
      - forecast bucket
  - name: Is Won
    expr: source.is_won
    display_name: Is Won
    comment: True if the opportunity was closed as won
    synonyms:
      - won
      - deal won
  - name: Is Closed
    expr: source.is_closed
    display_name: Is Closed
    comment: True if the opportunity is closed (won or lost)
    synonyms:
      - closed
  - name: Store ID
    expr: source.store_id
    display_name: Store
    comment: Identifier of the store where the transaction took place
    synonyms:
      - store
      - location
      - store location
  # ── Product dimensions (from product_catalog) ────────────────
  - name: Product Name
    expr: prod.product_name
    display_name: Product
    comment: Name of the sold product
    synonyms:
      - product
      - item
      - item name
  - name: Category
    expr: prod.category
    display_name: Product Category
    comment: High-level product category
    synonyms:
      - product category
      - product type
  - name: Subcategory
    expr: prod.subcategory
    display_name: Product Subcategory
    comment: Subcategory classification of the product
    synonyms:
      - product subcategory
      - sub-category
  - name: Brand
    expr: prod.brand
    display_name: Brand
    comment: Brand name of the sold product
    synonyms:
      - product brand
      - manufacturer brand
  - name: Supplier
    expr: prod.supplier_name
    display_name: Supplier
    comment: Supplier or vendor of the product
    synonyms:
      - vendor
      - supplier name
  # ── Customer / account dimensions (from account) ─────────────
  - name: Customer Name
    expr: acct.customer_name
    display_name: Customer
    comment: Name of the customer account
    synonyms:
      - account
      - account name
      - client
  - name: Account Type
    expr: acct.type
    display_name: Account Type
    comment: Account classification (e.g. Customer, Partner, Prospect)
    synonyms:
      - customer type
      - client type
  - name: Industry
    expr: acct.industry
    display_name: Industry
    comment: Industry vertical of the customer
    synonyms:
      - customer industry
      - vertical
      - sector
  - name: City
    expr: acct.billing_city
    display_name: City
    comment: Customer billing city
    synonyms:
      - billing city
      - customer city
  - name: State
    expr: acct.billing_state
    display_name: State
    comment: Customer billing state or province
    synonyms:
      - billing state
      - region
  - name: Country
    expr: acct.billing_country
    display_name: Country
    comment: Customer billing country
    synonyms:
      - billing country
      - geography
measures:
  # ── Volume measures ───────────────────────────────────────────
  - name: Transaction Count
    expr: COUNT(source.transaction_id)
    display_name: Transactions
    comment: Total number of sales transactions
    synonyms:
      - orders
      - sales count
      - number of orders
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Units Sold
    expr: SUM(source.quantity)
    display_name: Units Sold
    comment: Total number of product units sold
    synonyms:
      - quantity sold
      - total quantity
      - items sold
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Distinct Customers
    expr: COUNT(DISTINCT source.account_id)
    display_name: Active Customers
    comment: Number of distinct customer accounts with transactions
    synonyms:
      - customer count
      - unique customers
      - number of customers
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Distinct Products
    expr: COUNT(DISTINCT source.product_id)
    display_name: Products Sold
    comment: Number of distinct products transacted
    synonyms:
      - product count
      - unique products
      - number of products
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  # ── Revenue measures ──────────────────────────────────────────
  - name: Gross Revenue
    expr: SUM(source.gross_amount)
    display_name: Gross Revenue
    comment: Total revenue before discounts are applied
    synonyms:
      - gross sales
      - revenue before discount
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Total Discount
    expr: SUM(source.discount_amount)
    display_name: Total Discount
    comment: Total discount amount granted across all transactions
    synonyms:
      - discounts
      - total markdown
      - discount value
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Net Revenue
    expr: SUM(source.net_amount)
    display_name: Net Revenue
    comment: Total revenue after discounts — primary revenue KPI
    synonyms:
      - revenue
      - net sales
      - total revenue
      - sales
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Avg Order Value
    expr: MEASURE(`Net Revenue`) / MEASURE(`Transaction Count`)
    display_name: Avg Order Value
    comment: Average net revenue per transaction
    synonyms:
      - AOV
      - average transaction value
      - average basket
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
  - name: Avg Selling Price
    expr: SUM(source.selling_price) / COUNT(source.transaction_id)
    display_name: Avg Selling Price
    comment: Average selling price per transaction
    synonyms:
      - average price
      - average unit price
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
  - name: Discount Rate
    expr: SUM(source.discount_amount) / SUM(source.gross_amount)
    display_name: Discount Rate
    comment: Ratio of total discounts to gross revenue
    synonyms:
      - discount percentage
      - markdown rate
      - discount ratio
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1
  # ── Pipeline / opportunity measures ──────────────────────────
  - name: Pipeline Value
    expr: SUM(source.opportunity_amount)
    display_name: Pipeline Value
    comment: Total value of all opportunities in the pipeline
    synonyms:
      - opportunity value
      - total pipeline
      - pipeline amount
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Won Revenue
    expr: SUM(source.net_amount) FILTER (WHERE source.is_won = true)
    display_name: Won Revenue
    comment: Net revenue from closed-won opportunities only
    synonyms:
      - closed won revenue
      - won deal revenue
    format:
      type: currency
      currency_code: USD
      decimal_places:
        type: exact
        places: 2
      abbreviation: compact
  - name: Won Deal Count
    expr: COUNT(source.transaction_id) FILTER (WHERE source.is_won = true)
    display_name: Won Deals
    comment: Number of transactions/opportunities that were won
    synonyms:
      - won opportunities
      - closed won count
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Closed Deal Count
    expr: COUNT(source.transaction_id) FILTER (WHERE source.is_closed = true)
    display_name: Closed Deals
    comment: Total number of closed opportunities (won or lost)
    synonyms:
      - closed opportunities
      - total closed
    format:
      type: number
      decimal_places:
        type: exact
        places: 0
  - name: Win Rate
    expr: MEASURE(`Won Deal Count`) / MEASURE(`Closed Deal Count`)
    display_name: Win Rate
    comment: Percentage of closed deals that were won
    synonyms:
      - close rate
      - win percentage
      - win ratio
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1
  - name: Avg Win Probability
    expr: AVG(source.probability)
    display_name: Avg Win Probability
    comment: Average probability score across all opportunities
    synonyms:
      - win probability
      - close probability
      - average probability
    format:
      type: percentage
      decimal_places:
        type: exact
        places: 1
$$

In [0]:
%sql
-- Validate the metric view: key KPIs by Product Category and Sales Channel
SELECT
  `Category`,
  `Sales Channel`,
  MEASURE(`Transaction Count`)                         AS transactions,
  MEASURE(`Units Sold`)                                AS units_sold,
  MEASURE(`Net Revenue`)                               AS net_revenue,
  MEASURE(`Gross Revenue`)                             AS gross_revenue,
  MEASURE(`Total Discount`)                            AS total_discount,
  MEASURE(`Discount Rate`)                             AS discount_rate,
  MEASURE(`Avg Order Value`)                           AS avg_order_value,
  MEASURE(`Won Deal Count`)                            AS won_deals,
  MEASURE(`Win Rate`)                                  AS win_rate,
  MEASURE(`Distinct Customers`)                        AS customers
FROM retail.metric_view.retail_metrics_view
GROUP BY ALL
having transactions > 0 and units_sold > 0 and net_revenue > 0
ORDER BY net_revenue DESC
LIMIT 30